In [8]:
!pip install langchain langchain-huggingface langchain-community langchain-pinecone Pinecone

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv
import os
from getpass import getpass

load_dotenv()

# Fallback to interactive input if the API key isn't found in .env
api_key = os.getenv("PINECONE_API_KEY")
if not api_key:
    api_key = getpass("Please enter your Pinecone API Key: ")
    os.environ["PINECONE_API_KEY"] = api_key

pc = Pinecone(api_key=api_key)

Please enter your Pinecone API Key: ··········


In [11]:
index_name = "my-rag-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [15]:
free_embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-small-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
vector_store = PineconeVectorStore(
    index=index,
    embedding=free_embeddings
)

In [17]:
from langchain_core.documents import Document

doc1 = Document(
    page_content = "Tanjiro Kamado fights demons while holding on to his compassion and hope for his sister.",
    metadata = {"Anime" : "Demon Slayer"}
)
doc2 = Document(
    page_content = "Eren Yeager is driven by a fierce desire for freedom.",
    metadata = {"Anime" : "Attack on Titan"}
)
doc3 = Document(
    page_content = "Naruto Uzumaki never gives up on becoming Hokage and earning everyone’s respect.",
    metadata = {"Anime" : "Naruto"}
)
doc4 = Document(
    page_content = "Saitama can defeat almost anyone with one punch, but longs for a real challenge.",
    metadata = {"Anime" : "One Punch Man"}
)
doc4 = Document(
    page_content = "Ippo Makunouchi grows from a shy student into a determined boxer.",
    metadata = {"Anime" : "Hajime No Ippo"}
)
doc5 = Document(
    page_content = "Yuji Itadori risks himself to protect others after becoming Sukuna’s vessel.",
    metadata = {"Anime" : "Jujutsu Kaisen"}
)

In [18]:
docs = [doc1,doc2,doc3,doc4,doc5]

In [19]:
vector_store.add_documents(docs)

['6e3cc67d-431f-4682-892a-736b3ecb3b51',
 'd94644de-ea6c-4f7b-b204-8a9468cf1dcf',
 '531bf46e-acd3-406d-b68f-945f226a7b06',
 'd41be38a-3fda-4de7-908e-7b3f107a791c',
 '70459122-d08e-43a6-add0-1bac0e7b8384']

In [22]:
id1 = '6e3cc67d-431f-4682-892a-736b3ecb3b51'
id2 = 'd94644de-ea6c-4f7b-b204-8a9468cf1dcf'
id3 = '531bf46e-acd3-406d-b68f-945f226a7b06'
id4 = 'd41be38a-3fda-4de7-908e-7b3f107a791c'
id5 = '70459122-d08e-43a6-add0-1bac0e7b8384'


In [24]:
result = index.fetch(ids = [id1,id2,id3,id4,id5])

for id, record in result.vectors.items():
    print("Id: ",id)
    print("Embedding: ", record.values)
    print("Text: ", record.metadata["text"])
    print("Metadata: ", record.metadata["Anime"])

Id:  d94644de-ea6c-4f7b-b204-8a9468cf1dcf
Embedding:  [-0.00157525507, 0.0592358597, -0.0094338553, 0.0194673277, 0.0635858402, 0.0359063111, 0.0401813202, -0.0239891242, -0.00168094819, 0.0241492912, -0.038427107, 0.00814489461, -0.0106356731, -0.0281453039, 0.0665221214, 0.0310459677, 0.00721363723, 0.0453969836, -0.0273628738, 0.0762692392, -0.0527312756, -0.0459068157, 0.0498128198, -0.0754163116, 0.0260414854, -0.0155073404, 0.027900558, 0.0171167608, -0.0245271847, -0.0913391039, 0.0189678632, -0.0680164769, 0.0447519273, 0.0278201699, 0.0492785797, 0.0587570369, -0.0800686777, 0.0191874765, -0.0464110263, -0.0393161066, 0.0542920642, 0.0616290234, -0.0517104454, -0.0120992288, 0.0458714478, 0.0874237642, 0.0120496741, 0.0022758483, 0.00137112651, -0.0431789383, 0.000828080345, -0.0241745245, -0.018203117, -0.0205962379, 0.0393209904, 0.0262045301, 0.0337162763, 0.0094160093, -0.00468803849, -0.0208070595, -0.019799022, 0.0427978747, -0.0650436506, 0.00710277492, 0.0254157837, 0.

In [27]:
results = vector_store.similarity_search(
    query= "Which anime is about demons?",
    k = 2
)
for doc in results:
  print(doc.page_content)
  print(doc.metadata)

Tanjiro Kamado fights demons while holding on to his compassion and hope for his sister.
{'Anime': 'Demon Slayer'}
Naruto Uzumaki never gives up on becoming Hokage and earning everyone’s respect.
{'Anime': 'Naruto'}


In [35]:
results = vector_store.similarity_search_with_score(
    query='Which one is about killing monsters?',
    k=2
)
print(results)

[(Document(id='6e3cc67d-431f-4682-892a-736b3ecb3b51', metadata={'Anime': 'Demon Slayer'}, page_content='Tanjiro Kamado fights demons while holding on to his compassion and hope for his sister.'), 0.491291076), (Document(id='70459122-d08e-43a6-add0-1bac0e7b8384', metadata={'Anime': 'Jujutsu Kaisen'}, page_content='Yuji Itadori risks himself to protect others after becoming Sukuna’s vessel.'), 0.45862487)]


In [39]:
updated_doc1 = Document(
    page_content="When two strangers discover they can swap bodies across time and space, they must confront the heartbreaking truth that one of them is fighting against time itself to save the other from a tragedy that has already happened.",
    metadata = {"Anime": "Your Name"}
)
vector_store.add_documents(
    ids=[id1],
    documents=[updated_doc1]
)

['6e3cc67d-431f-4682-892a-736b3ecb3b51']

In [40]:
result = index.fetch(ids = [id1,id2,id3,id4,id5])

for id, record in result.vectors.items():
    print("Id: ",id)
    print("Embedding: ", record.values)
    print("Text: ", record.metadata["text"])
    print("Metadata: ", record.metadata["Anime"])

Id:  531bf46e-acd3-406d-b68f-945f226a7b06
Embedding:  [-0.014674834, 0.0244528335, -0.0249790233, -0.017087901, -0.0498642884, 0.0204020217, 0.0850668401, -0.0161588918, -0.0401455425, -0.0180756133, -0.0330392271, -0.0362409614, 0.0739831924, 2.38190751e-05, -0.041905582, -0.0148182306, -0.0351628065, 0.0565589294, -0.132818252, -0.0083860727, -0.00201649102, -0.058210142, 0.0166880507, 0.0035186864, 0.0933126807, -0.0130599374, 0.027012689, 0.0673620105, -0.013735339, -0.120554812, -0.00609077327, -0.135845065, 0.0288062114, 0.00888168067, -0.0553059727, 0.0116964839, -0.0112475, -0.00921841338, -0.0161688123, 0.000623178144, -0.0420098118, 0.125943363, 0.0225129705, -0.0788405, 0.0153301489, -0.0563342124, 0.0355797373, -0.0633833185, 0.0767978281, 0.0153031107, -0.0160447024, 0.000777344103, -0.0490948707, 0.078273572, 0.0314458646, -0.00967359636, 0.000583797402, -0.0239187386, 0.022991756, -0.0442535207, -0.0212677922, 0.047227636, -0.0910479128, 0.0161257479, 0.0562632494, -0.01